# 5d — Combined Climate Criticality

**Purpose.** Final step of the analysis: assemble every hazard, national-scale
disruption and local-accessibility result into a single per-road
*climate-criticality* index.

**Inputs** (all under `intermediate_results/`)
- Hazard exposure of the main network (`parquet/hazard_exposure/main_network_hazard_exposure.parquet`)
- Single-point-of-failure criticality results (`criticality_results.parquet`)
- Accessibility impact layers (`parquet/local_accessibility/*_impacts.parquet`)
- Climate-change layers: future flood return period and extreme-rainfall change (`parquet/hazard_exposure/`)

**Outputs**
- Maps: `figures/criticality_analysis_3panel.png`, `figures/climate_criticality_mean.png`
- ArcGIS `.gpkg` / `.lyrx` layers for the three sub-indices and the combined index
- Formatted multi-sheet workbook `results/Climate_Criticality_PuteviSrbije.xlsx`
- Geospatial export (`.parquet` + `.gpkg`, GeoPackage reprojected to EPSG:6316)

**Pipeline.** load & preprocess → clean → prepare metrics → deduplicate per
`oznaka_deo` → score (log → normalise → quintile → convex score → sub-indices
H/T/A → multiplicative combination) → plots / statistics / export.

The shared functions live in `utils/criticality_functions.py` (used by the
`5d` script too). This notebook is **independent of the config file**: paths and
the two analysis settings are hardcoded in the configuration cell below.


In [ ]:
import sys
import warnings
from pathlib import Path

import geopandas as gpd

# Make the project packages importable (utils/, src/)
BASE_DIR = Path.cwd().parent
sys.path.append(str(BASE_DIR))

from utils.criticality_functions import (
    clean_data,
    deduplicate_by_section,
    export_climate_criticality_excel,
    load_and_preprocess_criticality_data,
    plot_climate_criticality_components,
    plot_combined_climate_criticality,
    prepare_metrics,
    print_climate_criticality_statistics,
    save_climate_criticality_geospatial,
    score_climate_criticality,
)

warnings.simplefilter(action="ignore", category=FutureWarning)
warnings.simplefilter(action="ignore", category=RuntimeWarning)


In [ ]:
# --- Configuration (hardcoded; this notebook does not read network_config.py) ---

# Folder layout
intermediate_path = BASE_DIR / "intermediate_results"
parquet_path = intermediate_path / "parquet"
hazard_exposure_parquet = parquet_path / "hazard_exposure"
local_accessibility_parquet = parquet_path / "local_accessibility"
figure_path = BASE_DIR / "figures"
results_path = BASE_DIR / "results"
arcgis_results = results_path / "ArcGIS layers"
arcgis_gpkg = arcgis_results / "Geopackages"
for p in (figure_path, arcgis_results, arcgis_gpkg):
    p.mkdir(parents=True, exist_ok=True)

# Inputs
hazard_exposure_path = hazard_exposure_parquet / "main_network_hazard_exposure.parquet"
criticality_results_path = parquet_path / "travel_disruptions" / "criticality_results.parquet"
hospital_impacts_path = local_accessibility_parquet / "hospital_impacts.parquet"
factory_impacts_path = local_accessibility_parquet / "factory_impacts.parquet"
police_impacts_path = local_accessibility_parquet / "police_impacts.parquet"
fire_impacts_path = local_accessibility_parquet / "fire_impacts.parquet"
border_impacts_path = local_accessibility_parquet / "road_impacts.parquet"
port_impacts_path = local_accessibility_parquet / "port_impacts.parquet"
railway_impacts_path = local_accessibility_parquet / "rail_impacts.parquet"
future_floods_change_rp_path = hazard_exposure_parquet / "Future Floods change in RP.parquet"
future_rainfall_change_path = hazard_exposure_parquet / "change in maximum daily precipitation rcp 85 period 2.parquet"

# Outputs
climate_criticality_xlsx = results_path / "Climate_Criticality_PuteviSrbije.xlsx"
climate_criticality_parquet = intermediate_path / "Climate_Criticality_PuteviSrbije.parquet"
climate_criticality_gpkg = results_path / "Climate_Criticality_PuteviSrbije.gpkg"

# Analysis settings
output_crs = "EPSG:6316"
show_figures = True            # notebooks display figures inline
print_statistics = True
climate_hazards_only = True    # H built from climate-change hazards only (vs all hazards)
normalize_subindices = True    # CC_norm = norm(H) x (norm(T) + norm(A))  (vs CC_raw = H x (T + A))


In [ ]:
# 1. Load and spatially join every per-topic metric onto the road network
gdf = load_and_preprocess_criticality_data(
    hazard_exposure_path=hazard_exposure_path,
    criticality_results_path=criticality_results_path,
    hospital_impacts_path=hospital_impacts_path,
    factory_impacts_path=factory_impacts_path,
    police_impacts_path=police_impacts_path,
    fire_impacts_path=fire_impacts_path,
    border_impacts_path=border_impacts_path,
    port_impacts_path=port_impacts_path,
    railway_impacts_path=railway_impacts_path,
    future_floods_change_rp_path=future_floods_change_rp_path,
    future_rainfall_change_path=future_rainfall_change_path,
)
gdf.head()


In [ ]:
# 2. Clean, 3. standardise metric columns, 4. deduplicate per road section
gdf = clean_data(gdf)
gdf = prepare_metrics(gdf)
gdf = deduplicate_by_section(gdf)


In [ ]:
# 5. Score the combined climate-criticality index
gdf = score_climate_criticality(
    gdf,
    climate_hazards_only=climate_hazards_only,
    normalize_subindices=normalize_subindices,
)
gdf[["oznaka_deo", "kategorija", "H", "T", "A", "climate_criticality", "climate_criticality_class"]].head()


In [ ]:
# 6a. Three-panel map of the H / T / A sub-indices (+ ArcGIS layers)
plot_climate_criticality_components(
    gdf, figure_path, arcgis_gpkg, arcgis_results, show_figures=show_figures
)


In [ ]:
# 6b. Single map of the combined climate-criticality index (+ ArcGIS layer)
plot_combined_climate_criticality(
    gdf, figure_path, arcgis_gpkg, arcgis_results, show_figures=show_figures
)


In [ ]:
# Summary statistics
if print_statistics:
    print_climate_criticality_statistics(gdf)


In [ ]:
# Formatted multi-sheet Excel workbook
export_climate_criticality_excel(
    gdf, climate_criticality_xlsx,
    climate_hazards_only=climate_hazards_only,
    normalize_subindices=normalize_subindices,
)


In [ ]:
# Geospatial export: Parquet (working CRS) + GeoPackage (EPSG:6316)
save_climate_criticality_geospatial(
    gdf,
    parquet_path=climate_criticality_parquet,
    gpkg_path=climate_criticality_gpkg,
    output_crs=output_crs,
)
